<a href="https://colab.research.google.com/github/matiaslarregle/MegaSaludable_y_RepuestosFG/blob/main/Negocio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install dbfread
!pip install streamlit
import pandas as pd
from dbfread import DBF
import streamlit as st

archivo = "MAEART.DBF"
tabla = DBF(archivo, encoding='latin1')
archivo2 = "TRUBROS.DBF"
tabla2 = DBF(archivo2, encoding='latin1')
archivo4 = "TABTIT.DBF"
tabla4 = DBF(archivo4, encoding='latin1')

df = pd.DataFrame(iter(tabla))
df2 = pd.DataFrame(iter(tabla2))
df4 = pd.DataFrame(iter(tabla4))

df5 = pd.read_excel("CARR-SUR-Precios.xls", skiprows=13)
df6 = pd.read_excel("Lista de Precios MG.Gas 06-01-2025.xls", skiprows=3)

df6.columns = ["Index", "CODIGO", "Descripcion", "Precio"]
df5.columns = ["CODIGO", "Descripcion", "Moneda", "Precio", "IVA"]
df6 = df6.dropna(subset=["CODIGO"])
df5 = df5.dropna(subset=["CODIGO"])

df = df.merge(df2, left_on="RUBRO", right_on="CODIGO", how="left", suffixes=("", "_RUBRO"))
df = df.merge(df4, left_on="TITULO", right_on="CODIGO", how="left", suffixes=("", "_TITULO"))

df.rename(columns={"DESCRIP_RUBRO": "NOMBRE RUBRO", "DESCRIP_TITULO": "NOMBRE TITULO"}, inplace=True)

df.drop(columns=["CODIGO_RUBRO", "CODIGO_TITULO"], inplace=True)

columnas_a_conservar = ['NOMBRE RUBRO', 'NOMBRE TITULO', 'DESCRIP', 'CODIGO', 'UBICACION']

df = df[columnas_a_conservar]

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.6/9.6 MB 45.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 61.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.1/79.1 kB 4.6 MB/s eta 0:00:00


DBFNotFound: could not find file 'MAEART.DBF'

In [ ]:
columnas_a_conservar = ['CODIGO', 'Descripcion', 'Precio']

df5 = df5[columnas_a_conservar]
df6 = df6[columnas_a_conservar]

In [ ]:
df5['CODIGO'] = df5['CODIGO'].astype(float).astype(int).astype(str).str.lstrip('0')
df6['CODIGO'] = df6['CODIGO'].str.lstrip('0')

df5.rename(columns={'Descripcion': 'DESCRIPCION CARR SUR', 'Precio': 'PRECIO CARR SUR'}, inplace=True)
df6.rename(columns={'Descripcion': 'DESCRIPCION MG.GAS', 'Precio': 'PRECIO MG.GAS'}, inplace=True)

df_merged = pd.merge(df, df5, on=['CODIGO'], how='left', suffixes=('', '_CarrSur'))

df_merged = pd.merge(df_merged, df6, on=['CODIGO'], how='left', suffixes=('', '_MgGas'))

In [ ]:
df_merged

In [ ]:
!pip install streamlit
!pip install pyngrok
!pip install dbfread

# Crear el archivo de la app
app_code = """

import pandas as pd
from dbfread import DBF
import streamlit as st

# Cargar los archivos DBF
archivo = "MAEART.DBF"
archivo2 = "TRUBROS.DBF"
archivo4 = "TABTIT.DBF"

tabla = DBF(archivo, encoding='latin1')
tabla2 = DBF(archivo2, encoding='latin1')
tabla4 = DBF(archivo4, encoding='latin1')

# Convertir a DataFrame
df = pd.DataFrame(iter(tabla))
df2 = pd.DataFrame(iter(tabla2))
df4 = pd.DataFrame(iter(tabla4))

# Cargar archivos Excel
df5 = pd.read_excel("CARR-SUR-Precios(10).xls", skiprows=13)
df6 = pd.read_excel("Lista de Precios MG.Gas 06-01-2025.xls", skiprows=3)

# Renombrar columnas de los Excel
df6.columns = ["Index", "CODIGO", "Descripcion", "Precio"]
df5.columns = ["CODIGO", "Descripcion", "Moneda", "Precio", "IVA"]

# Eliminar filas con NaN innecesarios
df6 = df6.dropna(subset=["CODIGO"])
df5 = df5.dropna(subset=["CODIGO"])

# Unir datos con Rubros y Títulos
df = df.merge(df2, left_on="RUBRO", right_on="CODIGO", how="left", suffixes=("", "_RUBRO"))
df = df.merge(df4, left_on="TITULO", right_on="CODIGO", how="left", suffixes=("", "_TITULO"))

# Renombrar columnas
df.rename(columns={"DESCRIP_RUBRO": "NOMBRE RUBRO", "DESCRIP_TITULO": "NOMBRE TITULO"}, inplace=True)

# Eliminar columnas innecesarias
df.drop(columns=["CODIGO_RUBRO", "CODIGO_TITULO"], inplace=True)

# Especificar columnas a conservar
df = df[['NOMBRE RUBRO', 'NOMBRE TITULO', 'DESCRIP', 'CODIGO', 'UBICACION']]

# Filtrar columnas en df5 y df6
df5 = df5[['CODIGO', 'Descripcion', 'Precio']]
df6 = df6[['CODIGO', 'Descripcion', 'Precio']]

# Limpiar códigos de producto
df5['CODIGO'] = df5['CODIGO'].astype(float).astype(int).astype(str).str.lstrip('0')
df6['CODIGO'] = df6['CODIGO'].str.lstrip('0')

# Renombrar columnas de precios
df5.rename(columns={'Descripcion': 'Descripcion Carr Sur', 'Precio': 'Precio Carr Sur'}, inplace=True)
df6.rename(columns={'Descripcion': 'Descripcion Mg.Gas', 'Precio': 'Precio Mg.Gas'}, inplace=True)

# Merge con listas de precios
df_merged = df.merge(df5, on=['CODIGO'], how='left')
df_merged = df_merged.merge(df6, on=['CODIGO'], how='left')

# ---- INTERFAZ STREAMLIT ----
st.title("🔍 Sistema de Búsqueda de Productos")

# ---- FILTRO POR RUBRO ----
rubros_disponibles = df_merged["NOMBRE RUBRO"].dropna().unique().tolist()
rubro_seleccionado = st.sidebar.selectbox("📂 Seleccionar Rubro", [""] + rubros_disponibles)

# Filtrar DataFrame por rubro
df_filtrado = df_merged[df_merged["NOMBRE RUBRO"] == rubro_seleccionado] if rubro_seleccionado else df_merged.copy()

# ---- FILTRO POR TÍTULO ----
titulos_disponibles = df_filtrado["NOMBRE TITULO"].dropna().unique().tolist()
titulo_seleccionado = st.sidebar.selectbox("📌 Seleccionar Título", [""] + titulos_disponibles)

# Filtrar DataFrame por título
df_filtrado = df_filtrado[df_filtrado["NOMBRE TITULO"] == titulo_seleccionado] if titulo_seleccionado else df_filtrado

# ---- FILTRO POR DESCRIPCIÓN ----
descripcion_buscar = st.sidebar.text_input("📝 Buscar en Descripción")
if descripcion_buscar:
    df_filtrado = df_filtrado[df_filtrado["DESCRIP"].str.contains(descripcion_buscar, case=False, na=False)]

# ---- FILTRO POR CÓDIGO ----
codigo_buscar = st.sidebar.text_input("🔢 Buscar por Código")
if codigo_buscar:
    df_filtrado = df_filtrado[df_filtrado["CODIGO"].astype(str).str.contains(codigo_buscar, case=False, na=False)]

# ---- FILTRO POR UBICACIÓN ----
ubicacion_buscar = st.sidebar.text_input("📍 Buscar por Ubicación")
if ubicacion_buscar:
    df_filtrado = df_filtrado[df_filtrado["UBICACION"].astype(str).str.contains(ubicacion_buscar, case=False, na=False)]

# ---- INTERFAZ STREAMLIT ----
st.sidebar.header("➕ Agregar Nuevo Registro")

nuevo_codigo = st.sidebar.text_input("🔢 Código")

precio_carr_sur = df5.loc[df5["CODIGO"] == nuevo_codigo, "Precio Carr Sur"].values
precio_mg_gas = df6.loc[df6["CODIGO"] == nuevo_codigo, "Precio Mg.Gas"].values

precio_carr_sur = precio_carr_sur[0] if len(precio_carr_sur) > 0 else None
precio_mg_gas = precio_mg_gas[0] if len(precio_mg_gas) > 0 else None

nuevo_nombre_rubro = st.sidebar.text_input("📂 Nombre Rubro")
nuevo_nombre_titulo = st.sidebar.text_input("📌 Nombre Título")
nueva_descripcion = st.sidebar.text_input("📝 Descripción")
nueva_ubicacion = st.sidebar.text_input("📍 Ubicación")

if st.sidebar.button("✅ Agregar Registro"):
    if nuevo_codigo and nueva_descripcion:
        nuevo_registro = {
            "CODIGO": nuevo_codigo,
            "NOMBRE RUBRO": nuevo_nombre_rubro,
            "NOMBRE TITULO": nuevo_nombre_titulo,
            "DESCRIP": nueva_descripcion,
            "UBICACION": nueva_ubicacion,
            "Precio Carr Sur": precio_carr_sur,
            "Precio Mg.Gas": precio_mg_gas
        }

        df_nuevo = pd.DataFrame([nuevo_registro])
        df_filtrado = pd.concat([df_filtrado, df_nuevo], ignore_index=True)

        st.sidebar.success("🎉 Registro agregado con éxito!")
    else:
        st.sidebar.warning("⚠️ El Código y la Descripción son obligatorios.")

# ---- MOSTRAR RESULTADOS ----
st.write("📋 **Resultados:**")
st.dataframe(df_filtrado)

# ---- ENLACES DE BÚSQUEDA ----
st.write("🔗 **Búsqueda en otras fuentes:**")

busqueda = st.text_input("🔎 Ingrese un código o palabra clave para buscar en otras fuentes")

if busqueda:
    df_carr_sur_filtrado = df5[
        df5["CODIGO"].astype(str).str.contains(busqueda, case=False, na=False) |
        df5["Descripcion Carr Sur"].str.contains(busqueda, case=False, na=False)
    ]

    df_mg_gas_filtrado = df6[
        df6["CODIGO"].astype(str).str.contains(busqueda, case=False, na=False) |
        df6["Descripcion Mg.Gas"].str.contains(busqueda, case=False, na=False)
    ]

    busqueda_url = busqueda.replace(" ", "+")
    link_noroeste = f"🌎 [Buscar '{busqueda}' en Repuestos Noroeste](https://www.repuestos-noroeste.com.ar/articulos.php?b={busqueda_url})"

    if not df_carr_sur_filtrado.empty:
        st.write("📂 **Resultados en Carr Sur:**")
        st.dataframe(df_carr_sur_filtrado)
    else:
        st.warning("⚠️ No se encontraron resultados en Carr Sur.")

    if not df_mg_gas_filtrado.empty:
        st.write("📂 **Resultados en Metro Gas:**")
        st.dataframe(df_mg_gas_filtrado)
    else:
        st.warning("⚠️ No se encontraron resultados en Metro Gas.")

    # 🔹 Mostrar enlace a Noroeste
    st.markdown(f"- {link_noroeste}")


# ---- CALCULADORA DE PRECIOS ----
st.header("💰 Calculadora de Precios de Venta")

productos_disponibles = df_filtrado["DESCRIP"].dropna().unique().tolist()
producto_seleccionado = st.selectbox("🛒 Selecciona un producto", [""] + productos_disponibles)

precio_compra = None
if producto_seleccionado:
    precio_encontrado = df_filtrado[df_filtrado["DESCRIP"] == producto_seleccionado]["Precio Carr Sur"].values
    if len(precio_encontrado) > 0:
        precio_compra = precio_encontrado[0]

if precio_compra is not None:
    st.write(f"📌 **Precio de compra:** ${precio_compra:.2f}")

    margen = st.slider("📈 Margen de ganancia (%)", min_value=0, max_value=200, value=30, step=1)

    aplicar_iva = st.checkbox("➕ Aplicar IVA (21%)")

    precio_venta = precio_compra * (1 + margen / 100)
    if aplicar_iva:
        precio_venta *= 1.21

    st.write(f"💲 **Precio de venta sugerido:** ${precio_venta:.2f}")
else:
    st.warning("⚠️ Selecciona un producto válido para calcular el precio de venta.")



"""

with open('app.py', 'w') as f:
    f.write(app_code)

from pyngrok import ngrok

ngrok.set_auth_token('2raJPze5eiN8wEhhSoE6riZW9hm_4zdrSvBRGYdLJkCfEiyjc')

public_url = ngrok.connect('8501')
public_url

print("Tu aplicación de Streamlit está disponible en:", public_url)

!streamlit run app.py &

Tu aplicación de Streamlit está disponible en: NgrokTunnel: "https://db9d860b8a7f.ngrok-free.app" -> "http://localhost:8501"



  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.57.122.118:8501

